Training a RNN to model a sequence of words

In [1]:
import os
import torch
import random

import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from torch.distributions import Categorical

In [2]:
device = 'cpu'

In [3]:
# Hyperparameters 
hidden_size = 512   # size of hidden state
batch_size = 100    # size of the batch used for training
step_len = 200      # number of training samples in each stem
num_layers = 3      # number of layers in LSTM layer stack
lr = 0.002          # learning rate
num_steps = 50    # max number of training steps
gen_seq_len = 50    # length of generated sequence
load_chk = False    # load in pre-trained checkpoint for training
save_path = "word_rnn_model.pt"
# load_path = "word_rnn_model.pt"

In [4]:
def load_all_text_files_in_folder(path, max_files = 10000):
    corpus = ''
    # Find all files in the folder or subfolders
    for root, _, files in os.walk(path):
        for i, file in enumerate(files):
            # If the file is a text file
            if file.endswith(".txt") and i <= max_files:
                # Open the file and add the text to the corpus
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    text = f.read()
                    # Add text from file
                    corpus += text
                    # Add 'End of File' between documents
                    corpus += '\n EOF \n'
    return corpus

In [5]:
# Loading the dataset of Taylor Swift lyrics
data_path = "../data/tswiftalbums"
corpus = load_all_text_files_in_folder(data_path)
words = sorted(list(set(corpus.split())))
data_size, vocab_size = len(corpus.split()), len(words)

In [6]:
word_to_ix = { w:i for i,w in enumerate(words) }
ix_to_word = { i:w for i,w in enumerate(words) }

data = corpus.split()
for i, ch in enumerate(data):
    data[i] = word_to_ix[ch]

In [7]:
# Defining the network
class RNN(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, num_layers):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(input_size, input_size)
        self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers)
        self.decoder = nn.Linear(hidden_size, output_size)
    
    def forward(self, input_batch, hidden_state):
        embedding = self.embedding(input_batch)
        output, hidden_state = self.rnn(embedding, hidden_state)
        output = self.decoder(output)
        return output, (hidden_state[0].detach(), hidden_state[1].detach())

In [8]:
# Network and optimiser
# Create list of indexes that can be valid starting points for training
index_list = list(range(0, len(data) - step_len - 1))

# Conver data to torch tensor
data = torch.tensor(data).to(device)
data = torch.unsqueeze(data, dim=1)

# Create RNN class
rnn = RNN(vocab_size, vocab_size, hidden_size, num_layers).to(device)

# Define loss function and optimiser
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn.parameters(), lr=lr)

# Load in pretrained model if specified
if load_chk:
    checkpoint = torch.load(load_path)
    rnn.load_state_dict(checkpoint['state_dict'])

In [9]:
# sampling the data randomly
def get_training_batch_indicies(index_list, batch_size):
    # Get a batch of indicies to sample our data from
    input_batch_indicies = torch.tensor(np.array(random.sample(index_list, batch_size)))
    # Offset indicies for target batch by one
    target_batch_indicies = input_batch_indicies + 1
    return input_batch_indicies, target_batch_indicies

In [10]:
# Training the network
for step in range(1, num_steps):
    
    running_loss = 0
    hidden_state = None
    rnn.zero_grad()
    train_batch_indicies, target_batch_indicies = get_training_batch_indicies(index_list, batch_size)

    # Cycle through for a set number of consecutive iterations in the data
    for i in range(step_len):
        # Extract data batches from indicies
        input_batch = data[train_batch_indicies].squeeze()
        target_batch = data[target_batch_indicies].squeeze()
        
        # Forward pass
        # The following code is the same as calling rnn.forward(input_batch, hidden_state)
        output, hidden_state = rnn(input_batch, hidden_state)
        
        # Compute loss
        loss = loss_fn(output, target_batch)
        running_loss += loss.item() / step_len
        
        # Update weights of neural network
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        # Increment batch coordinates by 1
        train_batch_indicies = train_batch_indicies + 1
        target_batch_indicies = target_batch_indicies + 1

        
    # Print loss
    print('\n'+'-'*75)
    print(f"\nStep: {step} Loss: {running_loss}")

    # Create a dictionary for saving the model and data mappings
    save_dict = {}
    # Add the model weight parameters as a dictionary to our save_dict
    save_dict['state_dict'] = rnn.state_dict()
    # Add the idx_to_word and word_to_idx dicts to our save_dict
    save_dict['ix_to_word'] = ix_to_word
    save_dict['word_to_ix'] = word_to_ix
    # Save the dictionary to a file
    torch.save(save_dict, save_path)

    # Now lets generate a random generated text sample to print out,
    # we will do this without gradient tracking as we are not training
    with torch.no_grad():
        
        # Take a random index and reset the hidden state of the model
        rand_index = np.random.randint(data_size-1)
        input_batch = data[rand_index : rand_index+1]
        hidden_state = None
        
        # Iterate over our sequence length
        for i in range(gen_seq_len):
            # Forward pass
            output, hidden_state = rnn(input_batch, hidden_state)
            
            # Construct categorical distribution and sample a character
            output = F.softmax(torch.squeeze(output), dim=0)
            dist = Categorical(output)
            index = dist.sample()
            
            # Print the sampled character
            print(ix_to_word[index.item()], end=' ')
            
            # Next input is current output
            input_batch[0][0] = index.item()


---------------------------------------------------------------------------

Step: 1 Loss: 6.84688725233078
our catch think caught daydream a on," home [Bridge] one (Türkçe who him what I, back So the Tuesday and died Beautiful Phasa he's now" every Don't lonely Third To Prayin' afternoon and As Red like[Chorus] fine" to though It's run And weren't crime your break I not might your 
---------------------------------------------------------------------------

Step: 2 Loss: 6.525218057632453
ContributorsTranslationsTaylor thick leave now me Long chicks my People I'm with I'm hey, could yeah) you again And left what the Mind okay a times in dim I've made the (Can I be what I had midnight wish it is you Mr. last might me got the of And to 
---------------------------------------------------------------------------

Step: 3 Loss: 6.247267315387729
yeah-yeah Smoke first isn't it up), breaking in (Light my heart [Verse 2] do Oh, Don't have better When the heart bad tears la-la-la, it let the

Comments on the word rnn generator
I should have pre processed the data and used stop words to remove things like "contributors translations" "taylor swift" "verse" and other such comments from the lyrics dataset. 
2. The lyrics generated are actual words and not just random letters strung together, however the sentences are not always  comprehensible. The words generated are common words in a taylor swift song.

Training a RNN to generate lyrics modeling a sequence of characters

In [34]:
hidden_size = 512   # size of hidden state
batch_size = 600    # Size of the batch we use in training
step_len = 100      # number of training samples in each step
num_layers = 3      # number of layers in LSTM layer stack
lr = 0.002          # learning rate
num_steps = 30      # max number of training steps
gen_seq_len = 30    # length of generated sequence
load_chk = False    # load in pre-trained checkpoint for training
save_path = "char_rnn_model.pt"
# load_path = "char_rnn_model.pt"

In [29]:
device = 'cpu'

In [35]:
#checking if it fixes dataset error - it doesn't.
def load_all_text_files_in_folder(path, max_files = 10000):
    corpus = ''
    # Find all files in the folder or subfolders
    for root, _, files in os.walk(path):
        for i, file in enumerate(files):
            # If the file is a text file
            if file.endswith(".txt") and i <= max_files:
                # Open the file and add the text to the corpus
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    text = f.read()
                    # Add text from file
                    corpus += text
                    # Add 'End of File' between documents
                    corpus += '\n EOF \n'
    return corpus

In [36]:
# Loading the dataset
data_path = "../data/tswiftalbums/AllTooWell_10MinuteVersion__TaylorsVersion__FromtheVault_.txt"
corpus = open(data_path, 'r').read()
chars = sorted(list(set(corpus)))
data_size, vocab_size = len(corpus), len(chars)

The tswiftalbums folder is a directory containing multiple txt files and as such when used as a data path it returns an error. I have chosen one song to use for training the model. 

In [37]:
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

data = list(corpus)
for i, ch in enumerate(data):
    data[i] = char_to_ix[ch]

In [38]:
# DEFINING THE NEURAL NETWORK
class RNN(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, num_layers):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(input_size, input_size)
        self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers)
        self.decoder = nn.Linear(hidden_size, output_size)
    
    def forward(self, input_batch, hidden_state):
        embedding = self.embedding(input_batch)
        output, hidden_state = self.rnn(embedding, hidden_state)
        output = self.decoder(output)
        return output, (hidden_state[0].detach(), hidden_state[1].detach())

In [39]:
# ~etwork and optimiser
# Create list of indexes that can be valid starting points for training
index_list = list(range(0, len(data) - step_len - 1))

# Convert data to torch tensor
data = torch.tensor(data).to(device)
data = torch.unsqueeze(data, dim=1)

# Create RNN class
rnn = RNN(vocab_size, vocab_size, hidden_size, num_layers).to(device)

# Define loss function and optimiser
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn.parameters(), lr=lr)

# Load in pretrained model if specified
if load_chk:
    checkpoint = torch.load(load_path)
    rnn.load_state_dict(checkpoint['state_dict'])

In [40]:
# Sampling the data 
def get_training_batch_indicies(index_list, batch_size):
    # Get a batch of indicies to sample our data from
    input_batch_indicies = torch.tensor(np.array(random.sample(index_list, batch_size)))
    # Offset indicies for target batch by one
    target_batch_indicies = input_batch_indicies + 1
    return input_batch_indicies, target_batch_indicies

In [41]:
#Training the network
# Iterate through the number of steps defined earlier
for step in range(1, num_steps):
    
    running_loss = 0
    hidden_state = None
    rnn.zero_grad()
    train_batch_indicies, target_batch_indicies = get_training_batch_indicies(index_list, batch_size)

    
    # Cycle through for a set number of consecutive iterations in the data
    for i in range(step_len):
        # Extract data batches from indicies
        input_batch = data[train_batch_indicies].squeeze()
        target_batch = data[target_batch_indicies].squeeze()
        # Forward pass
        # The following code is the same as calling rnn.forward(input_batch, hidden_state)
        output, hidden_state = rnn(input_batch, hidden_state)
        
        # Compute loss
        loss = loss_fn(output, target_batch)
        running_loss += loss.item() / step_len
        
        # Update weights of neural network
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        # Increment batch coordinates by 1
        train_batch_indicies = train_batch_indicies + 1
        target_batch_indicies = target_batch_indicies + 1
        

        
    # Print loss
    print('\n'+'-'*75)
    print(f"\nStep: {step} Loss: {running_loss}")

    # Create a dictionary for saving the model and data mappings
    save_dict = {}
    # Add the model weight parameters as a dictionary to our save_dict
    save_dict['state_dict'] = rnn.state_dict()
    # Add the idx_to_char and char_to_idx dicts to our save_dict
    save_dict['ix_to_char'] = ix_to_char
    save_dict['char_to_ix'] = char_to_ix
    # Save the dictionary to a file
    torch.save(save_dict, save_path)

    # Now lets generate a random generated text sample to print out,
    # we will do this without gradient tracking as we are not training
    with torch.no_grad():
        
        # Take a random index and reset the hidden state of the model
        rand_index = np.random.randint(data_size-1)
        input_batch = data[rand_index : rand_index+1]
        hidden_state = None
        
        # Iterate over our sequence length
        for i in range(gen_seq_len):
            # Forward pass
            output, hidden_state = rnn(input_batch, hidden_state)
            
            # Construct categorical distribution and sample a character
            output = F.softmax(torch.squeeze(output), dim=0)
            dist = Categorical(output)
            index = dist.sample()
            
            # Print the sampled character
            print(ix_to_char[index.item()], end='')
            
            # Next input is current output
            input_batch[0][0] = index.item()


---------------------------------------------------------------------------

Step: 1 Loss: 2.7775298857688906
ereit thasao"uy waie'ivelloon]
---------------------------------------------------------------------------

Step: 2 Loss: 2.329932856559753
amd I'sh haret
Bu I I r t I se
---------------------------------------------------------------------------

Step: 3 Loss: 2.259259774684907
-be, oke I lerar catckfllllly 
---------------------------------------------------------------------------

Step: 4 Loss: 2.234377820491791
 I pid ' ion g n wemusadm t wa
---------------------------------------------------------------------------

Step: 5 Loss: 2.243618085384369
-Ph
I yower ct, we
All ce wang
---------------------------------------------------------------------------

Step: 6 Loss: 2.2324979376792915
d' clidd
[Ve in, f you tht dar
---------------------------------------------------------------------------

Step: 7 Loss: 2.216711564064026
e thevemyo pld Vemeorugheeru i
-----------------

Comments on character rnn lyrics generator
It is a much smaller dataset than with the word rnn generator so I don't expect the results to be comparable.
The lyrics generated are nonsensical strings of letters and not actual words. There is no semblance of a sentence structure like before. 


Using markov chains to generate lyrics

In [42]:
import os
import csv
import markovify

In [43]:
def load_single_text_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        corpus = f.read()
        return corpus

In [44]:
def load_all_text_files_in_folder(path, max_files = 10000):
    corpus = ''
    # Find all files in the folder or subfolders
    for root, _, files in os.walk(path):
        for i, file in enumerate(files):
            # If the file is a text file
            if file.endswith(".txt") and i <= max_files:
                # Open the file and add the text to the corpus
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    text = f.read()
                    # Add text from file
                    corpus += text
                    # Add new line
                    corpus += '\n'
    return corpus

In [45]:
def load_text_from_csv(csv_file, col_to_extract, delimeter):
    corpus = ''
    # Open csv file
    with open(csv_file, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile, delimiter=delimeter)
        for row in reader:
            # Check to see there is a column where we want to extract
            if len(row) >= col_to_extract:  
                # Get text from the specific column in the row
                text = row[col_to_extract]  
                # Add text to corpus
                corpus += text
                # Add a new line
                corpus += '\n'
    return corpus

In [48]:
data_path = '../data/tswiftalbums/AllTooWell_10MinuteVersion__TaylorsVersion__FromtheVault_.txt'
corpus = load_single_text_file(data_path)

In [57]:
text_model = markovify.Text(corpus, state_size=1)

for i in range(5):
    print(text_model.make_sentence())

Just between us, did the love affair maim you remember it all too well?
Just between us, did the love affair maim you remember it all too well?
Just between us, did the love affair maim you remember it all too well?
Just between us, did the love affair maim you remember it all too well?
Just between us, did the love affair maim you remember it all too well?


Comments on makov chain text generation
The markov chain using the same limited dataset was better than the character rnn at generating a coherent sentence, although there were still errors.
It was only able to generate one sentence repeatedly, regardless of adjustments in range. 
No lyrics were generated with a state size greater than 1

Conclusion: The word rnn generator was the closest in result to what I was trying to achieve.
# All code taken from classroom notebooks. 